# CNN-EXP-031 — Base-level ProCapNet-style backbone + regional 32/64-bp heads

Цель этого эксперимента — убрать BPE полностью и вернуться к base-resolution DNA.

Мы намеренно используем тот же base-level dilated CNN backbone, который уже применялся
в раннем CNN-эксперименте, но меняем выходную задачу:

```text
раньше:
base-level feature map
→ prediction на каждой bp

сейчас:
base-level feature map
→ regional pooling
→ 32-bp classifier
→ 64-bp auxiliary classifier
```

Главный objective:

```text
32 bp region contains >= 1 TSS ?
```

Дополнительный auxiliary objective:

```text
64 bp region contains >= 1 TSS ?
```

`64-bp` loss имеет небольшой вес, чтобы не дать coarse objective подавить 32-bp задачу:

```text
loss = loss32 + 0.25 * loss64
```

## Что здесь принципиально

- **NO BPE**
- **NO tokenizer**
- модель видит каждую базу на входе;
- backbone остаётся base-resolution;
- пространственное сжатие появляется только в самой regional head через pooling;
- 32/64 bp — это размер предсказываемого региона, а не размер токена;
- полный holdout оценивается теми же regional AP / TSS-recall метриками.

Это максимально близкий controlled test идеи:
`base-level sequence → learned CNN representation → coarse regional decision`.


In [1]:
from pathlib import Path
import copy
import gc
import math
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

import torch
import torch.nn as nn
import torch_directml

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    candidate
    for candidate in (
        CURRENT_DIR,
        *CURRENT_DIR.parents,
    )
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_DIR),
    )

from ml_project.epic_baseline import (
    load_baseline_sequences,
)

from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
)

from ml_project.epic_data import (
    load_split,
)

print("Project:", PROJECT_ROOT)


Project: D:\Projects\EPIC\EPIC


In [2]:
device = torch_directml.device()

print("DirectML device:", device)
print("GPU:", torch_directml.device_name(0))


DirectML device: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


In [3]:
# ============================================================
# DATA / WINDOW CONFIG
# ============================================================

split = load_split(
    PROJECT_ROOT
)

sequences, fasta_alphabet = (
    load_baseline_sequences(
        split
    )
)

train_contigs = tuple(
    sorted(
        set(
            split.intervals(
                "train"
            )[
                "contig"
            ].astype(str)
        )
    )
)

validation_contigs = tuple(
    sorted(
        set(
            split.intervals(
                "validation"
            )[
                "contig"
            ].astype(str)
        )
    )
)

assert set(
    train_contigs
).isdisjoint(
    validation_contigs
)

# Keep 2048-bp target so the regional grid is directly comparable
# with the recent regional experiments.
TARGET_LENGTH = 2048
CONTEXT_FLANK = 130

REGION_WIDTHS = (
    32,
    64,
)

PRIMARY_WIDTH = 32
AUX_WIDTH = 64
AUX_64_WEIGHT = 0.25

assert TARGET_LENGTH % 32 == 0
assert TARGET_LENGTH % 64 == 0

profile_config = (
    ProfileWindowConfig(
        target_length=TARGET_LENGTH,
        context_flank=CONTEXT_FLANK,
        positive_branch_fraction=0.5,
    )
)

train_generator = (
    ProfileWindowGenerator(
        prepared_split=split,
        sequences=sequences,
        part="train",
        config=profile_config,
        seed=42,
    )
)

validation_generator = (
    ProfileWindowGenerator(
        prepared_split=split,
        sequences=sequences,
        part="validation",
        config=profile_config,
        seed=42,
    )
)

display(split.summary)

print("Train contigs:", len(train_contigs))
print(
    "Validation contigs:",
    len(validation_contigs),
)
print(
    "Input length:",
    TARGET_LENGTH
    + 2 * CONTEXT_FLANK,
)
print(
    "Target length:",
    TARGET_LENGTH,
)
print(
    "Region widths:",
    REGION_WIDTHS,
)


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


Train contigs: 9
Validation contigs: 3
Input length: 2308
Target length: 2048
Region widths: (32, 64)


In [4]:
# ============================================================
# REAL BATCH CHECK
# ============================================================

smoke_batch = (
    train_generator
    .sample_batch(
        batch_size=4
    )
)

print(
    "x:",
    tuple(
        smoke_batch.x.shape
    ),
)

print(
    "x_both_strands:",
    tuple(
        smoke_batch
        .x_both_strands
        .shape
    ),
)

print(
    "target:",
    tuple(
        smoke_batch
        .target
        .shape
    ),
)

print(
    "count:",
    tuple(
        smoke_batch
        .count
        .shape
    ),
)

print(
    "mask:",
    tuple(
        smoke_batch
        .mask
        .shape
    ),
)

assert (
    smoke_batch
    .x_both_strands
    .shape[-1]
    == (
        TARGET_LENGTH
        + 2 * CONTEXT_FLANK
    )
)


x: (4, 5, 2308)
x_both_strands: (8, 5, 2308)
target: (4, 2, 2048)
count: (4, 2, 2048)
mask: (4, 2, 2048)


## Backbone

Здесь специально **не переписывается ранний CNN**.

Мы создаём тот же `create_cnn_presence_intensity()` и перехватываем feature map
непосредственно перед его старой `profile_head`.

Старая per-base head замораживается и используется только как технический проход,
а градиент нового regional loss идёт через перехваченные признаки обратно во весь backbone.

Это позволяет проверить именно гипотезу:

```text
тот же base-level backbone
+
другая regional head
```

без скрытой смены feature extractor.


In [5]:
# ============================================================
# REGIONAL HEAD ON TOP OF THE EXISTING BASE-LEVEL BACKBONE
# ============================================================


class RegionalPoolHead(
    nn.Module
):
    def __init__(
        self,
        in_channels,
    ):
        super().__init__()

        self.classifier = (
            nn.Sequential(
                nn.Conv1d(
                    2 * in_channels,
                    32,
                    1,
                ),
                nn.GELU(),
                nn.Conv1d(
                    32,
                    1,
                    1,
                ),
            )
        )

    def forward(
        self,
        target_features,
        width,
    ):
        B, C, L = (
            target_features.shape
        )

        if L % width != 0:
            raise ValueError(
                (L, width)
            )

        n_regions = (
            L // width
        )

        x = (
            target_features
            .reshape(
                B,
                C,
                n_regions,
                width,
            )
        )

        mean_features = (
            x.mean(
                dim=-1
            )
        )

        # RMS instead of max:
        # DirectML max backward caused scatter fallback/errors
        # in earlier regional experiments.
        rms_features = torch.sqrt(
            (
                x * x
            ).mean(
                dim=-1
            )
            + 1e-6
        )

        pooled = torch.cat(
            (
                mean_features,
                rms_features,
            ),
            dim=1,
        ).contiguous()

        return self.classifier(
            pooled
        )


class BaseLevelRegionalModel(
    nn.Module
):
    def __init__(self):
        super().__init__()

        self.base = (
            create_cnn_presence_intensity()
        )

        if not hasattr(
            self.base,
            "profile_head",
        ):
            raise AttributeError(
                "Expected base model to expose .profile_head"
            )

        self.backbone_channels = int(
            self.base
            .profile_head
            .in_channels
        )

        # Old per-base profile head is NOT part of the new objective.
        # Freeze its parameters; its input is captured by a hook.
        for parameter in (
            self.base
            .profile_head
            .parameters()
        ):
            parameter.requires_grad = False

        self.heads = nn.ModuleDict({
            str(width): RegionalPoolHead(
                self.backbone_channels
            )
            for width in REGION_WIDTHS
        })

        self._captured_features = None

        self._feature_hook = (
            self.base
            .profile_head
            .register_forward_pre_hook(
                self._capture_backbone_features
            )
        )

    def _capture_backbone_features(
        self,
        module,
        inputs,
    ):
        self._captured_features = (
            inputs[0]
        )

    def forward(
        self,
        x,
    ):
        self._captured_features = None

        # Run the known DirectML-safe base model.
        # Its old outputs are intentionally ignored.
        _ = self.base(
            x
        )

        features = (
            self._captured_features
        )

        self._captured_features = None

        if features is None:
            raise RuntimeError(
                "Backbone feature hook did not fire."
            )

        expected_length = (
            TARGET_LENGTH
            + 2 * CONTEXT_FLANK
        )

        if (
            features.shape[-1]
            != expected_length
        ):
            raise ValueError(
                (
                    features.shape,
                    expected_length,
                )
            )

        target_features = (
            features[
                :,
                :,
                CONTEXT_FLANK:
                CONTEXT_FLANK
                + TARGET_LENGTH
            ]
            .contiguous()
        )

        outputs = {}

        for width in REGION_WIDTHS:
            outputs[
                width
            ] = (
                self.heads[
                    str(width)
                ](
                    target_features,
                    width,
                )
            )

        return outputs


def create_model():
    return BaseLevelRegionalModel()


model_cpu = create_model()

trainable_params = sum(
    p.numel()
    for p in model_cpu.parameters()
    if p.requires_grad
)

frozen_params = sum(
    p.numel()
    for p in model_cpu.parameters()
    if not p.requires_grad
)

print(
    "Backbone channels:",
    model_cpu.backbone_channels,
)

print(
    "Trainable params:",
    f"{trainable_params:,}",
)

print(
    "Frozen old-head params:",
    f"{frozen_params:,}",
)

del model_cpu


Backbone channels: 32
Trainable params: 63,298
Frozen old-head params: 66


In [6]:
# ============================================================
# TARGETS / LOSS / FORWARD HELPERS
# ============================================================


def make_region_targets_cpu(
    exact_target,
    base_weight,
    width,
):
    if (
        exact_target.device.type
        != "cpu"
    ):
        raise ValueError(
            "Expected CPU exact_target"
        )

    if (
        base_weight.device.type
        != "cpu"
    ):
        raise ValueError(
            "Expected CPU base_weight"
        )

    B = exact_target.shape[0]

    n_regions = (
        TARGET_LENGTH
        // width
    )

    target = (
        exact_target
        .float()
        .reshape(
            B,
            1,
            n_regions,
            width,
        )
        .amax(
            dim=-1
        )
    )

    valid = (
        (
            base_weight > 0
        )
        .reshape(
            B,
            1,
            n_regions,
            width,
        )
        .all(
            dim=-1
        )
        .float()
    )

    return (
        target.contiguous(),
        valid.contiguous(),
    )


def stable_region_bce(
    logits,
    targets,
    valid,
    *,
    max_pos_weight=32.0,
):
    targets = targets.to(
        dtype=logits.dtype
    )

    valid = valid.to(
        dtype=logits.dtype
    )

    with torch.no_grad():
        pos = (
            targets * valid
        ).sum()

        neg = (
            (1.0 - targets)
            * valid
        ).sum()

        pos_weight = torch.clamp(
            neg
            / torch.clamp(
                pos,
                min=1.0,
            ),
            min=1.0,
            max=max_pos_weight,
        )

    positive_term = (
        torch.clamp(
            -logits,
            min=0,
        )
        + torch.log1p(
            torch.exp(
                -torch.abs(
                    logits
                )
            )
        )
    )

    negative_term = (
        torch.clamp(
            logits,
            min=0,
        )
        + torch.log1p(
            torch.exp(
                -torch.abs(
                    logits
                )
            )
        )
    )

    element_loss = (
        targets
        * pos_weight
        * positive_term
        + (
            1.0 - targets
        )
        * negative_term
    )

    denom = torch.clamp(
        valid.sum(),
        min=1.0,
    )

    loss = (
        element_loss
        * valid
    ).sum() / denom

    return (
        loss,
        float(
            pos_weight
            .detach()
            .cpu()
            .item()
        ),
    )


def forward_both(
    model,
    x_both,
    batch_size,
):
    outputs_all = model(
        x_both
    )

    result = {}

    for width in REGION_WIDTHS:
        logits = outputs_all[
            width
        ]

        result[
            width
        ] = (
            logits[
                :batch_size
            ].contiguous(),
            logits[
                batch_size:
            ].contiguous(),
        )

    return result


In [7]:
# ============================================================
# ONE REAL DIRECTML BACKWARD STEP
# ============================================================

torch.manual_seed(123)

smoke_model = (
    create_model()
    .to(device)
)

smoke_optimizer = (
    DirectMLAdam(
        (
            p
            for p in (
                smoke_model
                .parameters()
            )
            if p.requires_grad
        ),
        lr=1e-3,
    )
)

smoke_model.train()

x_gpu = (
    smoke_batch
    .x_both_strands
    .to(device)
)

smoke_optimizer.zero_grad(
    set_to_none=True
)

outputs = forward_both(
    smoke_model,
    x_gpu,
    batch_size=4,
)

losses = {}

for width in REGION_WIDTHS:
    pp, mp = outputs[
        width
    ]

    plus_target_r, plus_valid_r = (
        make_region_targets_cpu(
            smoke_batch.plus_target,
            smoke_batch.plus_loss_weight,
            width,
        )
    )

    minus_target_r, minus_valid_r = (
        make_region_targets_cpu(
            smoke_batch
            .minus_target_reverse,
            smoke_batch
            .minus_loss_weight_reverse,
            width,
        )
    )

    plus_loss, plus_pw = (
        stable_region_bce(
            pp,
            plus_target_r.to(
                device
            ),
            plus_valid_r.to(
                device
            ),
        )
    )

    minus_loss, minus_pw = (
        stable_region_bce(
            mp,
            minus_target_r.to(
                device
            ),
            minus_valid_r.to(
                device
            ),
        )
    )

    losses[width] = (
        plus_loss
        + minus_loss
    ) / 2.0

smoke_loss = (
    losses[32]
    + AUX_64_WEIGHT
    * losses[64]
)

smoke_loss.backward()
smoke_optimizer.step()

print(
    "Smoke loss32:",
    float(
        losses[32]
        .detach()
        .cpu()
    ),
)

print(
    "Smoke loss64:",
    float(
        losses[64]
        .detach()
        .cpu()
    ),
)

print(
    "Smoke total:",
    float(
        smoke_loss
        .detach()
        .cpu()
    ),
)

print(
    "DirectML backward smoke-test: OK"
)

del smoke_model
del smoke_optimizer
gc.collect()


Smoke loss32: 1.0248217582702637
Smoke loss64: 1.090017557144165
Smoke total: 1.2973260879516602
DirectML backward smoke-test: OK


1113

In [8]:
# ============================================================
# TRAIN — 10K ARCHITECTURE SCREEN
# ============================================================

STEPS = 10_000
SAVE_EVERY = 2_500
BATCH_SIZE = 4
LOG_EVERY = 100
SEED = 42

RUN_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP031_BASELEVEL_PROCAPNET_REGIONAL_32_64"
    / "run_001"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HISTORY_PATH = (
    RUN_DIR
    / "training_history.csv"
)

torch.manual_seed(
    SEED
)

generator = (
    ProfileWindowGenerator(
        prepared_split=split,
        sequences=sequences,
        part="train",
        config=profile_config,
        seed=SEED,
    )
)

model = (
    create_model()
    .to(device)
)

optimizer = (
    DirectMLAdam(
        (
            p
            for p in (
                model.parameters()
            )
            if p.requires_grad
        ),
        lr=1e-3,
    )
)

model.train()

history = []

ema32 = None
ema64 = None
ema_total = None

seen = 0
started = time.perf_counter()


def tree_to_cpu(
    value
):
    if torch.is_tensor(
        value
    ):
        return (
            value.detach()
            .cpu()
            .clone()
        )

    if isinstance(
        value,
        dict,
    ):
        return {
            key: tree_to_cpu(
                item
            )
            for key, item
            in value.items()
        }

    if isinstance(
        value,
        list,
    ):
        return [
            tree_to_cpu(
                item
            )
            for item in value
        ]

    if isinstance(
        value,
        tuple,
    ):
        return tuple(
            tree_to_cpu(
                item
            )
            for item in value
        )

    return value


def checkpoint_path(
    step
):
    return (
        RUN_DIR
        / (
            f"checkpoint_step_"
            f"{step:06d}.pt"
        )
    )


def save_checkpoint(
    step
):
    path = (
        checkpoint_path(
            step
        )
    )

    checkpoint = {
        "experiment": (
            "CNN-EXP-031"
        ),
        "architecture": (
            "baselevel_existing_dilated_cnn_regional_32_64"
        ),
        "step": int(
            step
        ),
        "model_state_dict": {
            k: (
                v.detach()
                .cpu()
                .clone()
            )
            for k, v
            in model
            .state_dict()
            .items()
        },
        "optimizer_state_dict": (
            tree_to_cpu(
                optimizer
                .state_dict()
            )
        ),
        "generator_rng_state": (
            copy.deepcopy(
                generator
                .rng
                .bit_generator
                .state
            )
        ),
        "torch_rng_state": (
            torch.get_rng_state()
        ),
        "target_length": (
            TARGET_LENGTH
        ),
        "context_flank": (
            CONTEXT_FLANK
        ),
        "region_widths": (
            REGION_WIDTHS
        ),
        "aux_64_weight": (
            AUX_64_WEIGHT
        ),
        "batch_size": (
            BATCH_SIZE
        ),
        "seed": (
            SEED
        ),
        "training_budget": (
            STEPS
        ),
        "ema_32": float(
            ema32
        ),
        "ema_64": float(
            ema64
        ),
        "ema_total": float(
            ema_total
        ),
        "elapsed_minutes": float(
            (
                time.perf_counter()
                - started
            )
            / 60.0
        ),
    }

    torch.save(
        checkpoint,
        path,
    )

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False,
    )

    print()
    print(
        f"=== CHECKPOINT SAVED @ {step:,} ==="
    )
    print(
        "EMA32:",
        round(
            float(
                ema32
            ),
            6,
        ),
    )
    print(
        "EMA64:",
        round(
            float(
                ema64
            ),
            6,
        ),
    )
    print(
        "Path:",
        path,
    )
    print()


with warnings.catch_warnings(
    record=True
) as caught:

    warnings.simplefilter(
        "always"
    )

    for step in range(
        1,
        STEPS + 1,
    ):
        batch = (
            generator
            .sample_batch(
                batch_size=(
                    BATCH_SIZE
                )
            )
        )

        x_gpu = (
            batch
            .x_both_strands
            .to(device)
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        outputs = (
            forward_both(
                model,
                x_gpu,
                batch_size=(
                    BATCH_SIZE
                ),
            )
        )

        width_losses = {}
        width_pws = {}

        for width in REGION_WIDTHS:
            pp, mp = (
                outputs[
                    width
                ]
            )

            (
                plus_target_r,
                plus_valid_r,
            ) = (
                make_region_targets_cpu(
                    batch.plus_target,
                    batch.plus_loss_weight,
                    width,
                )
            )

            (
                minus_target_r,
                minus_valid_r,
            ) = (
                make_region_targets_cpu(
                    batch
                    .minus_target_reverse,
                    batch
                    .minus_loss_weight_reverse,
                    width,
                )
            )

            (
                plus_loss,
                plus_pw,
            ) = stable_region_bce(
                pp,
                plus_target_r.to(
                    device
                ),
                plus_valid_r.to(
                    device
                ),
            )

            (
                minus_loss,
                minus_pw,
            ) = stable_region_bce(
                mp,
                minus_target_r.to(
                    device
                ),
                minus_valid_r.to(
                    device
                ),
            )

            width_losses[
                width
            ] = (
                plus_loss
                + minus_loss
            ) / 2.0

            width_pws[
                width
            ] = (
                plus_pw
                + minus_pw
            ) / 2.0

        loss32 = (
            width_losses[32]
        )

        loss64 = (
            width_losses[64]
        )

        total_loss = (
            loss32
            + AUX_64_WEIGHT
            * loss64
        )

        total_value = (
            total_loss.item()
        )

        if not math.isfinite(
            total_value
        ):
            raise FloatingPointError(
                f"Non-finite loss "
                f"at step {step}"
            )

        total_loss.backward()
        optimizer.step()

        value32 = (
            loss32.item()
        )

        value64 = (
            loss64.item()
        )

        if ema_total is None:
            ema32 = value32
            ema64 = value64
            ema_total = (
                total_value
            )
        else:
            ema32 = (
                0.95 * ema32
                + 0.05 * value32
            )

            ema64 = (
                0.95 * ema64
                + 0.05 * value64
            )

            ema_total = (
                0.95 * ema_total
                + 0.05
                * total_value
            )

        seen += int(
            batch.mask.sum()
        )

        elapsed = (
            time.perf_counter()
            - started
        )

        row = {
            "step": step,
            "loss_32": value32,
            "loss_64": value64,
            "total_loss": (
                total_value
            ),
            "ema_32": ema32,
            "ema_64": ema64,
            "ema_total": (
                ema_total
            ),
            "pos_weight_32": (
                width_pws[32]
            ),
            "pos_weight_64": (
                width_pws[64]
            ),
            "positions_per_second": (
                seen / elapsed
            ),
        }

        history.append(
            row
        )

        if (
            step == 1
            or step
            % LOG_EVERY
            == 0
        ):
            print(
                f"step={step:6d} "
                f"loss32="
                f"{value32:.4f} "
                f"ema32="
                f"{ema32:.4f} "
                f"loss64="
                f"{value64:.4f} "
                f"ema64="
                f"{ema64:.4f} "
                f"total="
                f"{total_value:.4f} "
                f"positions/s="
                f"{seen / elapsed:,.0f}"
            )

        if (
            step % SAVE_EVERY == 0
            or step == STEPS
        ):
            save_checkpoint(
                step
            )


fallback = [
    str(w.message)
    for w in caught
    if "fall back"
    in str(
        w.message
    ).lower()
]

if fallback:
    raise RuntimeError(
        "DirectML fallback: "
        + " | ".join(
            fallback
        )
    )

pd.DataFrame(
    history
).to_csv(
    HISTORY_PATH,
    index=False,
)

print()
print(
    "Elapsed min:",
    round(
        (
            time.perf_counter()
            - started
        )
        / 60,
        2,
    ),
)

print(
    "Final EMA32:",
    ema32,
)

print(
    "Final EMA64:",
    ema64,
)

print(
    "Final checkpoint:",
    checkpoint_path(
        STEPS
    ),
)


step=     1 loss32=1.0349 ema32=1.0349 loss64=1.1505 ema64=1.1505 total=1.3225 positions/s=242,844
step=   100 loss32=1.0711 ema32=1.0633 loss64=1.1585 ema64=1.1200 total=1.3607 positions/s=287,395
step=   200 loss32=1.1246 ema32=1.0004 loss64=1.1559 ema64=1.0647 total=1.4135 positions/s=279,366
step=   300 loss32=1.2207 ema32=0.9851 loss64=1.1267 ema64=1.0596 total=1.5024 positions/s=281,638
step=   400 loss32=1.1730 ema32=0.9235 loss64=1.2562 ema64=1.0507 total=1.4871 positions/s=287,321
step=   500 loss32=0.6399 ema32=0.9545 loss64=0.8704 ema64=1.0431 total=0.8575 positions/s=292,544
step=   600 loss32=0.7110 ema32=0.9068 loss64=0.8214 ema64=1.0085 total=0.9164 positions/s=294,401
step=   700 loss32=0.8346 ema32=0.9187 loss64=1.1412 ema64=1.0336 total=1.1199 positions/s=295,696
step=   800 loss32=1.1604 ema32=0.9019 loss64=1.1932 ema64=1.0335 total=1.4587 positions/s=296,792
step=   900 loss32=0.7061 ema32=0.8527 loss64=0.7394 ema64=0.9124 total=0.8909 positions/s=296,302
step=  100

In [9]:
# ============================================================
# FULL HOLDOUT TILE INDEX
# ============================================================

validation_tiles = (
    validation_generator
    .tile_index
    .copy()
)

print(
    "Tiles:",
    len(
        validation_tiles
    ),
)

print(
    "Position-strand:",
    int(
        validation_tiles[
            "allowed_position_strand"
        ].sum()
    ),
)

print(
    "Positives:",
    int(
        validation_tiles[
            "positive_positions"
        ].sum()
    ),
)

print(
    "Contigs:",
    validation_tiles[
        "contig"
    ].nunique(),
)


Tiles: 24700
Position-strand: 83807486
Positives: 56205
Contigs: 3


In [10]:
# ============================================================
# REGIONAL EVALUATION HELPERS
# ============================================================


TOP_REGION_FRACTIONS = (
    0.01,
    0.02,
    0.05,
    0.10,
    0.20,
    0.50,
)

TARGET_TSS_RECALLS = (
    0.90,
    0.95,
    0.99,
)


def sigmoid_numpy(
    x
):
    x = np.asarray(
        x,
        dtype=np.float64,
    )

    out = np.empty_like(
        x
    )

    positive = (
        x >= 0
    )

    out[
        positive
    ] = (
        1.0
        / (
            1.0
            + np.exp(
                -x[
                    positive
                ]
            )
        )
    )

    exp_x = np.exp(
        x[
            ~positive
        ]
    )

    out[
        ~positive
    ] = (
        exp_x
        / (
            1.0
            + exp_x
        )
    )

    return out


def exact_average_precision(
    scores,
    labels,
):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    labels = np.asarray(
        labels,
        dtype=bool,
    )

    positives = int(
        labels.sum()
    )

    if positives == 0:
        return np.nan

    order = np.argsort(
        -scores,
        kind="mergesort",
    )

    y = (
        labels[
            order
        ]
        .astype(
            np.int64
        )
    )

    tp = np.cumsum(
        y,
        dtype=np.int64,
    )

    precision = (
        tp
        / np.arange(
            1,
            y.size + 1,
            dtype=np.float64,
        )
    )

    return float(
        precision[
            y == 1
        ].sum()
        / positives
    )


def region_arrays_from_batch(
    score_plus,
    score_minus_reverse,
    mask,
    counts,
    width,
):
    B = mask.shape[0]

    n_regions = (
        TARGET_LENGTH
        // width
    )

    plus_score = (
        score_plus[
            :,
            0,
            :,
        ]
    )

    minus_score = (
        score_minus_reverse[
            :,
            0,
            ::-1,
        ]
    )

    score = np.stack(
        (
            plus_score,
            minus_score,
        ),
        axis=1,
    )

    m = mask.reshape(
        B,
        2,
        n_regions,
        width,
    )

    c = counts.reshape(
        B,
        2,
        n_regions,
        width,
    )

    valid = (
        m.all(
            axis=-1
        )
    )

    tss_count = (
        (
            (c > 0)
            & m
        )
        .sum(
            axis=-1
        )
        .astype(
            np.int32
        )
    )

    positive = (
        tss_count > 0
    )

    return (
        score,
        positive,
        tss_count,
        valid,
    )


def summarize_region_ranking(
    width,
    scores,
    positive,
    tss_count,
):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    positive = np.asarray(
        positive,
        dtype=bool,
    )

    tss_count = np.asarray(
        tss_count,
        dtype=np.int64,
    )

    n = int(
        scores.size
    )

    positive_regions = int(
        positive.sum()
    )

    total_tss = int(
        tss_count.sum()
    )

    prevalence = (
        positive_regions / n
    )

    ap = (
        exact_average_precision(
            scores,
            positive,
        )
    )

    ap_lift = (
        ap / prevalence
        if prevalence > 0
        else np.nan
    )

    order = np.argsort(
        -scores,
        kind="mergesort",
    )

    y_region = (
        positive[
            order
        ]
        .astype(
            np.int64
        )
    )

    y_tss = (
        tss_count[
            order
        ]
        .astype(
            np.int64
        )
    )

    cum_positive_regions = (
        np.cumsum(
            y_region,
            dtype=np.int64,
        )
    )

    cum_tss = np.cumsum(
        y_tss,
        dtype=np.int64,
    )

    ranking_rows = []

    for fraction in (
        TOP_REGION_FRACTIONS
    ):
        k = max(
            1,
            int(
                np.ceil(
                    fraction
                    * n
                )
            ),
        )

        k = min(
            k,
            n,
        )

        selected_positive_regions = int(
            cum_positive_regions[
                k - 1
            ]
        )

        covered_tss = int(
            cum_tss[
                k - 1
            ]
        )

        ranking_rows.append({
            "region_width_bp": (
                width
            ),
            "selection_type": (
                "top_fraction"
            ),
            "target": fraction,
            "selected_regions": k,
            "selected_fraction": (
                k / n
            ),
            "regional_precision": (
                selected_positive_regions
                / k
            ),
            "positive_region_recall": (
                selected_positive_regions
                / positive_regions
                if positive_regions > 0
                else np.nan
            ),
            "covered_tss": (
                covered_tss
            ),
            "tss_recall": (
                covered_tss
                / total_tss
                if total_tss > 0
                else np.nan
            ),
        })

    for recall_target in (
        TARGET_TSS_RECALLS
    ):
        required = (
            recall_target
            * total_tss
        )

        idx = int(
            np.searchsorted(
                cum_tss,
                required,
                side="left",
            )
        )

        idx = min(
            idx,
            n - 1,
        )

        k = idx + 1

        selected_positive_regions = int(
            cum_positive_regions[
                idx
            ]
        )

        covered_tss = int(
            cum_tss[
                idx
            ]
        )

        ranking_rows.append({
            "region_width_bp": (
                width
            ),
            "selection_type": (
                "target_tss_recall"
            ),
            "target": (
                recall_target
            ),
            "selected_regions": k,
            "selected_fraction": (
                k / n
            ),
            "regional_precision": (
                selected_positive_regions
                / k
            ),
            "positive_region_recall": (
                selected_positive_regions
                / positive_regions
                if positive_regions > 0
                else np.nan
            ),
            "covered_tss": (
                covered_tss
            ),
            "tss_recall": (
                covered_tss
                / total_tss
                if total_tss > 0
                else np.nan
            ),
        })

    summary = {
        "region_width_bp": (
            width
        ),
        "regions": n,
        "positive_regions": (
            positive_regions
        ),
        "regional_prevalence": (
            prevalence
        ),
        "total_tss": (
            total_tss
        ),
        "regional_average_precision": (
            ap
        ),
        "regional_ap_lift": (
            ap_lift
        ),
    }

    return (
        summary,
        ranking_rows,
    )


def evaluate_regional_model(
    model,
    tiles,
    batch_size=8,
    progress_every=500,
):
    score_chunks = {
        width: []
        for width
        in REGION_WIDTHS
    }

    positive_chunks = {
        width: []
        for width
        in REGION_WIDTHS
    }

    tss_count_chunks = {
        width: []
        for width
        in REGION_WIDTHS
    }

    positions = 0
    exact_positives = 0

    started = (
        time.perf_counter()
    )

    was_training = (
        model.training
    )

    model.eval()

    try:
        with warnings.catch_warnings(
            record=True
        ) as caught:

            warnings.simplefilter(
                "always"
            )

            with torch.no_grad():
                for (
                    batch_no,
                    start,
                ) in enumerate(
                    range(
                        0,
                        len(tiles),
                        batch_size,
                    ),
                    1,
                ):
                    selected = (
                        tiles.iloc[
                            start:
                            start
                            + batch_size
                        ]
                    )

                    batch = (
                        validation_generator
                        .make_batch(
                            selected
                        )
                    )

                    B = len(
                        selected
                    )

                    x_gpu = (
                        batch
                        .x_both_strands
                        .to(device)
                    )

                    outputs = (
                        forward_both(
                            model,
                            x_gpu,
                            batch_size=B,
                        )
                    )

                    mask = (
                        batch.mask
                        .numpy()
                        .astype(
                            bool
                        )
                    )

                    counts = (
                        batch.count
                        .numpy()
                    )

                    positions += int(
                        mask.sum()
                    )

                    exact_positives += int(
                        (
                            mask
                            & (
                                counts > 0
                            )
                        ).sum()
                    )

                    for width in REGION_WIDTHS:
                        pp, mp = (
                            outputs[
                                width
                            ]
                        )

                        plus_score = (
                            sigmoid_numpy(
                                pp.detach()
                                .cpu()
                                .numpy()
                            )
                        )

                        minus_score_reverse = (
                            sigmoid_numpy(
                                mp.detach()
                                .cpu()
                                .numpy()
                            )
                        )

                        (
                            score,
                            positive,
                            tss_count,
                            valid,
                        ) = (
                            region_arrays_from_batch(
                                plus_score,
                                minus_score_reverse,
                                mask,
                                counts,
                                width,
                            )
                        )

                        score_chunks[
                            width
                        ].append(
                            score[
                                valid
                            ].astype(
                                np.float32
                            )
                        )

                        positive_chunks[
                            width
                        ].append(
                            positive[
                                valid
                            ].astype(
                                np.bool_
                            )
                        )

                        tss_count_chunks[
                            width
                        ].append(
                            tss_count[
                                valid
                            ].astype(
                                np.int16
                            )
                        )

                    finished = (
                        start
                        + len(
                            selected
                        )
                        >= len(
                            tiles
                        )
                    )

                    if (
                        batch_no == 1
                        or batch_no
                        % progress_every
                        == 0
                        or finished
                    ):
                        elapsed = (
                            time.perf_counter()
                            - started
                        )

                        print(
                            f"tiles="
                            f"{start + len(selected):>5}"
                            f"/{len(tiles)} "
                            f"positions="
                            f"{positions:>9} "
                            f"positions/s="
                            f"{positions / elapsed:,.0f}"
                        )

        fallback = tuple(
            str(
                w.message
            )
            for w in caught
            if "fall back"
            in str(
                w.message
            ).lower()
        )

    finally:
        model.train(
            was_training
        )

    if fallback:
        raise RuntimeError(
            "DirectML fallback: "
            + " | ".join(
                fallback
            )
        )

    assert (
        positions
        == int(
            tiles[
                "allowed_position_strand"
            ].sum()
        )
    )

    assert (
        exact_positives
        == int(
            tiles[
                "positive_positions"
            ].sum()
        )
    )

    summary_rows = []
    ranking_rows = []

    for width in REGION_WIDTHS:
        scores = np.concatenate(
            score_chunks[
                width
            ]
        )

        positive = np.concatenate(
            positive_chunks[
                width
            ]
        )

        tss_count = (
            np.concatenate(
                tss_count_chunks[
                    width
                ]
            )
            .astype(
                np.int64
            )
        )

        summary, rows = (
            summarize_region_ranking(
                width,
                scores,
                positive,
                tss_count,
            )
        )

        summary_rows.append(
            summary
        )

        ranking_rows.extend(
            rows
        )

    summary_df = (
        pd.DataFrame(
            summary_rows
        )
        .sort_values(
            "region_width_bp"
        )
        .reset_index(
            drop=True
        )
    )

    ranking_df = (
        pd.DataFrame(
            ranking_rows
        )
        .sort_values(
            [
                "region_width_bp",
                "selection_type",
                "target",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    elapsed = (
        time.perf_counter()
        - started
    )

    return (
        summary_df,
        ranking_df,
        elapsed,
        positions / elapsed,
    )


In [11]:
# ============================================================
# FULL HOLDOUT @ 10K
# ============================================================

CHECKPOINT_STEP = 10_000

CHECKPOINT_PATH = (
    checkpoint_path(
        CHECKPOINT_STEP
    )
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

assert (
    checkpoint[
        "architecture"
    ]
    == (
        "baselevel_existing_dilated_cnn_regional_32_64"
    )
)

assert tuple(
    checkpoint[
        "region_widths"
    ]
) == REGION_WIDTHS

validation_model = (
    create_model()
    .to(device)
)

validation_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

print()
print(
    "=" * 72
)

print(
    f"FULL HOLDOUT @ "
    f"{CHECKPOINT_STEP:,} STEPS"
)

print(
    "=" * 72
)

(
    holdout_summary,
    holdout_ranking,
    validation_seconds,
    validation_speed,
) = evaluate_regional_model(
    validation_model,
    validation_tiles,
    batch_size=8,
    progress_every=500,
)

holdout_summary.insert(
    0,
    "step",
    CHECKPOINT_STEP,
)

holdout_ranking.insert(
    0,
    "step",
    CHECKPOINT_STEP,
)

display(
    holdout_summary
)

print()
print(
    "=== TOP 10% REGIONS ==="
)

display(
    holdout_ranking[
        (
            holdout_ranking[
                "selection_type"
            ]
            == "top_fraction"
        )
        & np.isclose(
            holdout_ranking[
                "target"
            ],
            0.10,
        )
    ]
)

print()
print(
    "=== REGIONS NEEDED FOR 95% TSS ==="
)

display(
    holdout_ranking[
        (
            holdout_ranking[
                "selection_type"
            ]
            == "target_tss_recall"
        )
        & np.isclose(
            holdout_ranking[
                "target"
            ],
            0.95,
        )
    ]
)

SUMMARY_PATH = (
    RUN_DIR
    / "holdout_summary_10k.csv"
)

RANKING_PATH = (
    RUN_DIR
    / "holdout_ranking_10k.csv"
)

holdout_summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

holdout_ranking.to_csv(
    RANKING_PATH,
    index=False,
)

print()
print(
    "Validation minutes:",
    round(
        validation_seconds
        / 60.0,
        2,
    ),
)

print(
    "Validation positions/s:",
    f"{validation_speed:,.0f}",
)

print(
    "Saved:",
    SUMMARY_PATH,
)

print(
    "Saved:",
    RANKING_PATH,
)



FULL HOLDOUT @ 10,000 STEPS
tiles=    8/24700 positions=    23524 positions/s=1,160,315
tiles= 4000/24700 positions= 13735464 positions/s=3,611,915
tiles= 8000/24700 positions= 27763582 positions/s=3,669,398
tiles=12000/24700 positions= 41193676 positions/s=3,643,274
tiles=16000/24700 positions= 54896176 positions/s=3,647,007
tiles=20000/24700 positions= 67811662 positions/s=3,593,506
tiles=24000/24700 positions= 81734314 positions/s=3,626,316
tiles=24700/24700 positions= 83807486 positions/s=3,618,064


,step,region_width_bp,regions,positive_regions,regional_prevalence,total_tss,regional_average_precision,regional_ap_lift
0,10000,32,2560380,27982,0.010929,55730,0.111679,10.218717
1,10000,64,1250248,24027,0.019218,55105,0.140835,7.328377



=== TOP 10% REGIONS ===


,step,region_width_bp,selection_type,target,selected_regions,selected_fraction,regional_precision,positive_region_recall,covered_tss,tss_recall
6,10000,32,top_fraction,0.1,256038,0.1,0.062682,0.573547,39450,0.707877
15,10000,64,top_fraction,0.1,125025,0.1,0.098260,0.511300,37411,0.678904



=== REGIONS NEEDED FOR 95% TSS ===


,step,region_width_bp,selection_type,target,selected_regions,selected_fraction,regional_precision,positive_region_recall,covered_tss,tss_recall
1,10000,32,target_tss_recall,0.95,1176681,0.459573,0.021895,0.920699,52944,0.950009
10,10000,64,target_tss_recall,0.95,628548,0.502739,0.034877,0.912390,52350,0.950005



Validation minutes: 0.4
Validation positions/s: 3,479,611
Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP031_BASELEVEL_PROCAPNET_REGIONAL_32_64\run_001\holdout_summary_10k.csv
Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP031_BASELEVEL_PROCAPNET_REGIONAL_32_64\run_001\holdout_ranking_10k.csv


## Как читать EXP031

Главное сравнение — **32-bp head** с текущими 10k regional моделями.

Смотреть:

```text
regional_average_precision
TSS recall @ top10%
regions needed for 95% TSS recall
```

64-bp head здесь вспомогательная:

- она показывает, легче ли base-level backbone находить более широкую окрестность;
- её loss имеет вес только `0.25`;
- если 32 bp ухудшается относительно 32-only baseline, следующий ablation — убрать 64-bp auxiliary head.

Если base-level модель уже на 10k заметно превосходит BPE-модели, это сильный аргумент
двигаться дальше без tokenizer и масштабировать именно ProCapNet/BPNet-style backbone.
